Celda 1 — Imports y carga de datos

In [2]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import grad, jit
import matplotlib.pyplot as plt

data = np.load("datos_6111.npz")
X, y, t = data["X"], data["y"], data["t"]   # X: (24,256), y: (24,), t: (256,)
print(X.shape, y.shape, t.shape)

K = 15
N = X.shape[1]   # 256

ModuleNotFoundError: No module named 'jax'

Parte 1 — Capa Fourier 
Construcción de W1​ (Fourier)

In [ ]:
def construir_W1_fourier(t, K):
    """
    Filas 0..K-1   -> sin(2*pi*k*t)
    Filas K..2K-1  -> cos(2*pi*k*t)
    Retorna matriz (2K, N)
    """
    ks = jnp.arange(1, K + 1)[:, None]      # (K,1)
    tt = t[None, :]                          # (1,N)
    senos  = jnp.sin(2 * jnp.pi * ks * tt)   # (K,N)
    cosenos = jnp.cos(2 * jnp.pi * ks * tt)  # (K,N)
    return jnp.concatenate([senos, cosenos], axis=0)  # (2K,N)

W1_fourier = construir_W1_fourier(jnp.array(t), K)
print(W1_fourier.shape)  # (30, 256)

In [1]:
señal_prueba = X[0]
proy = W1_fourier @ señal_prueba   # (30,)

fft_ref = np.fft.rfft(señal_prueba)  # complejo, longitud N/2+1

# Comparamos componente k=1..K (ignorando k=0, la DC)
coef_coseno_manual = proy[K:] * (2 / N)   # filas de coseno
coef_seno_manual   = proy[:K] * (2 / N)   # filas de seno

coef_coseno_fft = fft_ref.real[1:K+1] * (2 / N)
coef_seno_fft   = -fft_ref.imag[1:K+1] * (2 / N)   # signo por convención

plt.figure(figsize=(7,4))
plt.plot(coef_coseno_manual, 'o-', label="Capa Fourier (coseno)")
plt.plot(coef_coseno_fft, 'x--', label="np.fft (coseno)")
plt.xlabel("Índice de modo k")
plt.ylabel("Coeficiente")
plt.title("Verificación: capa Fourier vs np.fft.rfft")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'X' is not defined